# Resume Screening System — Jupyter Analysis Notebook
**Task 3 | spaCy · NLTK · scikit-learn | TF-IDF + Cosine Similarity**


In [ ]:
import sys
sys.path.insert(0, '../src')
from resume_screener import *
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print('Modules loaded successfully ✅')

## 1. Text Preprocessing Demo

In [ ]:
preprocessor = TextPreprocessor()

sample_text = '''
I have 3+ years of experience in Machine Learning, Python, TensorFlow and NLP.
Worked at Google on BERT fine-tuning for text classification tasks.
Contact: john.doe@email.com | +91-9876543210 | https://github.com/johndoe
'''

print('=== Raw Text ===\n', sample_text)
print('=== After Noise Removal ===\n', preprocessor.remove_noise(sample_text))
tokens = preprocessor.tokenize_and_filter(preprocessor.remove_noise(sample_text))
print('\n=== Tokens (after stopword removal) ===\n', tokens)
lemmatized = preprocessor.lemmatize(tokens)
print('\n=== Lemmatized ===\n', lemmatized)
print('\n=== Final Cleaned Text ===\n', preprocessor.clean(sample_text))

## 2. Skill Extraction Demo

In [ ]:
nlp = initialize_nlp_resources()
parser = ResumeParser(nlp=nlp)

resume_text = '''
Experienced in Python, TensorFlow, PyTorch, scikit-learn, NLP, BERT,
AWS, Docker, Kubernetes, SQL, MongoDB, REST API, Git, Deep Learning.
3 years experience in data science and machine learning.
B.Tech CSE from IIT Bombay.
'''

skills = parser.extract_skills(resume_text)
exp = parser.extract_experience_years(resume_text)
edu = parser.extract_education(resume_text)

print(f'Extracted Skills ({len(skills)}): {skills}')
print(f'Experience: {exp} years')
print(f'Education: {edu}')

## 3. TF-IDF Vectorization Visualization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

job_desc = 'python machine learning scikit-learn tensorflow NLP data science'
resumes = [
    'python tensorflow scikit-learn NLP deep learning',
    'java spring boot react angular frontend',
    'python pandas numpy data analysis matplotlib',
    'machine learning python scikit-learn AWS Docker',
]

vectorizer = TfidfVectorizer()
corpus = [job_desc] + resumes
tfidf_matrix = vectorizer.fit_transform(corpus)
similarities = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()

print('TF-IDF Cosine Similarities:')
for i, sim in enumerate(similarities):
    print(f'  Resume {i+1}: {sim:.4f}')

# Plot
plt.figure(figsize=(7, 4))
bars = plt.bar([f'R{i+1}' for i in range(len(resumes))], similarities,
               color=['green' if s > 0.5 else 'orange' for s in similarities])
plt.title('TF-IDF Cosine Similarity vs Job Description')
plt.ylabel('Similarity Score')
plt.ylim(0, 1)
for bar, sim in zip(bars, similarities):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{sim:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Full Pipeline Run

In [ ]:
# Import sample data from main.py
import sys; sys.path.insert(0, '..')
from main import SAMPLE_RESUMES, JOB_DESCRIPTIONS

pipeline = ResumeScreeningPipeline(output_dir='../outputs')
scores = pipeline.run(
    jd_title='Data Scientist',
    jd_text=JOB_DESCRIPTIONS['Data Scientist'],
    resume_data=SAMPLE_RESUMES,
    generate_charts=True
)

# Show results as DataFrame
import pandas as pd
df = pd.DataFrame([{
    'Rank': s.rank, 'Name': s.name, 'Score': s.final_score,
    'TF-IDF': s.tfidf_similarity, 'Skill': s.skill_match_score,
    'Gap%': s.skill_gap_percentage, 'Recommendation': s.recommendation
} for s in scores])
df.set_index('Rank', inplace=True)
df